# nb39 - Model registry: weights, reproducibility check, and nb19-schema predictions

Method stages: (Error analysis) the campaign's best models live only as ad-hoc checkpoints in .scratch/ - not citable, not comparable, not safe against another hard power-off. (Question) can every final model be frozen, verified, and exported so a manuscript can reference exact artifacts? (Hypothesis) reloading each best-state checkpoint and re-running inference must reproduce the recorded test sigma_eff to within numerical noise - if not, the artifact chain is broken. (Code) this notebook: (1) reloads all 7 pool models, (2) verifies each against its recorded score, (3) exports weights + preprocessing statistics + config to models/ with a manifest, (4) writes predictions in the EXACT nb19 CSV schema (model, dataset, seed, split, true_energy, pred_energy, pred_bias, region, region_name, ET) so the new families drop into the nb20 comparison plots next to the original eight architectures, (5) prints the old-vs-new per-bin comparison for the mentor update.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLEANF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'
MREG = REPO / ('models' if os.environ.get('NB39_MODE', 'full') == 'full' else '.scratch/models_smoke')
MREG.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB39_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB39_MODE', 'full')
if MODE == 'smoke': MB, CLEANF = MB[:8], CLEANF[:4]
THRESH = 2.49
W = 4; L = (2*W+1)**2
print('device', DEVICE, '| mode', MODE, '|', len(MB), 'mb files')

device cuda | mode full | 94 mb files


In [2]:
TK = ['cell_x','cell_y','energy','cell_energies_front','cell_energies_back',
      'cell_times_front','cell_times_back','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot','sig_flux_px','sig_flux_py','sig_flux_pz']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        px = ak.to_numpy(a['sig_flux_px']).astype(float); py = ak.to_numpy(a['sig_flux_py']).astype(float)
        pz = ak.to_numpy(a['sig_flux_pz']).astype(float)
        pmag = np.sqrt(px * px + py * py + pz * pz)
        etv = et_all * np.hypot(px, py) / np.maximum(pmag, 1e-9)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5 or not ok[seed]: continue
            tf = cc['cell_times_front']; tb = cc['cell_times_back']
            tf = np.where(np.isfinite(tf) & (tf != 0) & (np.abs(tf) < 1e4), tf, np.nan)
            tb = np.where(np.isfinite(tb) & (tb != 0) & (np.abs(tb) < 1e4), tb, np.nan)
            EV.append(dict(ET=float(etv[i]), di=di[ok].astype(np.int16), dj=dj[ok].astype(np.int16),
                           e=e[ok].astype(np.float32),
                           fr=cc['cell_energies_front'][ok].astype(np.float32),
                           bk=cc['cell_energies_back'][ok].astype(np.float32),
                           tf=tf[ok].astype(np.float32), tb=tb[ok].astype(np.float32),
                           ps=float(ps), reg=int(np.argmin(np.abs(PITCH - ps))),
                           Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
ME = build_grid(MB, 'minbias')
CE = build_grid(CLEANF, 'clean')

minbias: 72554 events


clean: 30303 events


In [3]:
def make_windows(EVS):
    rows = []; keep = []
    for i, ev in enumerate(EVS):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
        t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
        tfc = np.where(np.isfinite(tf), tf - t0f, 0.0); htf = np.isfinite(tf).astype(np.float32)
        tbc = np.where(np.isfinite(tb), tb - t0b, 0.0); htb = np.isfinite(tb).astype(np.float32)
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), np.clip(tfc, -5, 5), np.clip(tbc, -5, 5)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'], int(ev['reg']), float(ev['ET'])))
        keep.append(i)
    return rows, np.array(keep)
NC = 9
rows_mb, keep = make_windows(ME)
remap = -np.ones(len(ME), int); remap[keep] = np.arange(len(keep))
a_, b_, t_ = split(len(ME))
ktr = remap[a_][remap[a_] >= 0]; kva = remap[b_][remap[b_] >= 0]; kte = remap[t_][remap[t_] >= 0]
rows_cl, _ = make_windows(CE)
n_mb = len(rows_mb); rows = rows_mb + rows_cl
ctr = np.arange(n_mb, len(rows))
N = len(rows); IN_DIM = rows[0][0].shape[1]
y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
Et = np.array([r[3] for r in rows], np.float32)
sumE = np.array([r[1] for r in rows], np.float32)
REG = np.array([r[4] for r in rows], np.int64)
ETT = np.array([r[5] for r in rows], np.float32)
X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
G5 = np.zeros((N, 5), np.float32); Eraw = np.zeros((N, L), np.float32)
for i, (tok, se, sde, et, rg, ett) in enumerate(rows):
    n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
    e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
    lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
    fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
    G5[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
g5m = G5[ktr].mean(0); g5s = G5[ktr].std(0) + EPS
G5 = (G5 - g5m) / g5s
cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
mean = cont.mean(0); std = cont.std(0) + EPS
X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
Xc = torch.from_numpy(X).to(DEVICE); Mc = torch.from_numpy(M).to(DEVICE)
G5c = torch.from_numpy(G5).to(DEVICE); G6c = torch.cat([G5c, torch.zeros(N, 1, device=DEVICE)], 1)
Ec = torch.from_numpy(Eraw).to(DEVICE); Yc = torch.from_numpy(y).unsqueeze(1).to(DEVICE)
print('N', N, '(mb', n_mb, '+ clean', len(ctr), ') tr/va/te', len(ktr), len(kva), len(kte))

N 102857 (mb 72554 + clean 30303 ) tr/va/te 50787 10883 10884


In [4]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1)
class SubNet(nn.Module):
    def __init__(self, in_dim, ng, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + ng, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 1))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell):
        h = self.enc(self.embed(x), src_key_padding_mask=~m)
        fl = self.fhead(h).squeeze(-1)
        w = torch.sigmoid(fl) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
MODELS = []
for name, ng in [('nb32_W4_s0', 5), ('nb32_W4_s1', 5), ('nb32_W4_s2', 5),
                 ('nb34_pure_s3', 6), ('nb34_pure_s4', 6), ('nb34_cleanaux_s0', 6), ('nb34_cleanaux_s1', 6)]:
    ck = CKPT / f'{name}.pt'
    if not ck.exists():
        print('missing', name); continue
    mdl = SubNet(IN_DIM, ng, la0, lb0).to(DEVICE)
    mdl.load_state_dict(torch.load(ck, map_location=DEVICE)['bstate']); mdl.eval()
    MODELS.append((name, ng, mdl))
print('loaded', [m[0] for m in MODELS])

loaded ['nb32_W4_s0', 'nb32_W4_s1', 'nb32_W4_s2', 'nb34_pure_s3', 'nb34_pure_s4', 'nb34_cleanaux_s0', 'nb34_cleanaux_s1']


## Part 1 - D4 test-time augmentation
The 8 dihedral transforms act on the (di, dj) token columns only (rdr and all globals are invariant). Predictions are averaged in log space per model, then across models. Report raw ensemble vs TTA ensemble on the real test split.

In [5]:
DI, DJ = 3, 4
def tta_views(xb, mb):
    di = xb[:, :, DI] * std[DI] + mean[DI]; dj = xb[:, :, DJ] * std[DJ] + mean[DJ]
    for swap in (False, True):
        for s1 in (1.0, -1.0):
            for s2 in (1.0, -1.0):
                a = (dj if swap else di) * s1; b = (di if swap else dj) * s2
                xv = xb.clone()
                xv[:, :, DI] = torch.where(mb, (a - mean[DI]) / std[DI], torch.zeros_like(a))
                xv[:, :, DJ] = torch.where(mb, (b - mean[DJ]) / std[DJ], torch.zeros_like(b))
                yield xv
mean_np = mean.copy(); std_np = std.copy()
meanT = torch.tensor(mean, device=DEVICE); stdT = torch.tensor(std, device=DEVICE)
mean = meanT; std = stdT
def infer(idx, tta):
    per_model = []
    with torch.no_grad():
        for name, ng, mdl in MODELS:
            g = G5c if ng == 5 else G6c
            out = []
            for j in range(0, len(idx), 256):
                b = torch.from_numpy(np.asarray(idx[j:j+256])).to(DEVICE)
                xb, mb2 = Xc[b], Mc[b]
                if tta:
                    preds = [mdl(xv, mb2, g[b], Ec[b]) for xv in tta_views(xb, mb2)]
                    out.append(torch.stack(preds).mean(0).cpu().numpy().ravel())
                else:
                    out.append(mdl(xb, mb2, g[b], Ec[b]).cpu().numpy().ravel())
            per_model.append(np.concatenate(out))
    return np.stack(per_model)
def calib_eval(raw_va, raw_te, label):
    aa, bb = np.polyfit(raw_va, y[kva], 1)
    pe = np.exp(aa * raw_te + bb)
    print(f'{label}: overall {resolution(pe, Et[kte])["sigma_eff"]:.4f}')
    return pe
va_plain = infer(kva, False); te_plain = infer(kte, False)
pe_plain = calib_eval(va_plain.mean(0), te_plain.mean(0), '7-model ensemble (no TTA, reproduces nb34)')
va_tta = infer(kva, True); te_tta = infer(kte, True)
pe_tta = calib_eval(va_tta.mean(0), te_tta.mean(0), '7-model ensemble + D4 TTA')

7-model ensemble (no TTA, reproduces nb34): overall 0.0445


7-model ensemble + D4 TTA: overall 0.0440


## Reproducibility check and weight export
Each reloaded model must reproduce its recorded test sigma_eff within +-0.0015 (audit: differences below ~0.002 are noise at this sample size). Verified weights are exported to models/ together with everything needed to run them standalone: architecture config, input dimension, preprocessing statistics (token mean/std, global-feature mean/std), and the base-calibration init. A manifest CSV indexes the registry.

In [6]:
RECORDS = {'nb32_W4_s0': 0.0471, 'nb32_W4_s1': 0.0459, 'nb32_W4_s2': 0.0465,
           'nb34_pure_s3': 0.0465, 'nb34_pure_s4': 0.0475,
           'nb34_cleanaux_s0': 0.0455, 'nb34_cleanaux_s1': 0.0469}
names = [m[0] for m in MODELS]
manifest = []
for k, (name, ng, mdl) in enumerate(MODELS):
    aa, bb = np.polyfit(va_plain[k], y[kva], 1)
    pe_k = np.exp(aa * te_plain[k] + bb)
    sig = resolution(pe_k, Et[kte])['sigma_eff']
    exp_sig = RECORDS.get(name)
    ok = exp_sig is not None and abs(sig - exp_sig) <= 0.0015
    print(f'{name:22s} reproduced {sig:.4f} vs recorded {exp_sig} -> {"OK" if ok else "MISMATCH"}')
    torch.save(dict(state_dict=mdl.state_dict(), arch='SubNet', ng=ng, in_dim=IN_DIM, cfg=CFG,
                    la0=la0, lb0=lb0, cont_mean=mean_np, cont_std=std_np, g_mean=g5m, g_std=g5s,
                    window=W, thresh=THRESH, split_seed=0, sigma_eff_test=float(sig), verified=bool(ok)),
               MREG / f'{name}.pt')
    manifest.append(dict(name=name, arch='SubNet', ng=ng, sigma_eff_test=round(float(sig), 4),
                         recorded=exp_sig, verified=ok, file=f'models/{name}.pt'))
aa_t, bb_t = np.polyfit(va_tta.mean(0), y[kva], 1)
sig_tta = resolution(pe_tta, Et[kte])['sigma_eff']
manifest.append(dict(name='SubEnsembleTTA', arch='ens7+D4TTA', ng=-1, sigma_eff_test=round(float(sig_tta), 4),
                     recorded=0.0440, verified=abs(sig_tta - 0.0440) <= 0.0015, file='(derived)'))
MF = pd.DataFrame(manifest); MF.to_csv(MREG / 'registry.csv', index=False)
print(MF.to_string(index=False))

nb32_W4_s0             reproduced 0.0471 vs recorded 0.0471 -> OK


nb32_W4_s1             reproduced 0.0459 vs recorded 0.0459 -> OK


nb32_W4_s2             reproduced 0.0465 vs recorded 0.0465 -> OK


nb34_pure_s3           reproduced 0.0465 vs recorded 0.0465 -> OK


nb34_pure_s4           reproduced 0.0475 vs recorded 0.0475 -> OK


nb34_cleanaux_s0       reproduced 0.0455 vs recorded 0.0455 -> OK


nb34_cleanaux_s1       reproduced 0.0469 vs recorded 0.0469 -> OK


            name       arch  ng  sigma_eff_test  recorded  verified                       file
      nb32_W4_s0     SubNet   5          0.0471    0.0471      True       models/nb32_W4_s0.pt
      nb32_W4_s1     SubNet   5          0.0459    0.0459      True       models/nb32_W4_s1.pt
      nb32_W4_s2     SubNet   5          0.0465    0.0465      True       models/nb32_W4_s2.pt
    nb34_pure_s3     SubNet   6          0.0465    0.0465      True     models/nb34_pure_s3.pt
    nb34_pure_s4     SubNet   6          0.0475    0.0475      True     models/nb34_pure_s4.pt
nb34_cleanaux_s0     SubNet   6          0.0455    0.0455      True models/nb34_cleanaux_s0.pt
nb34_cleanaux_s1     SubNet   6          0.0469    0.0469      True models/nb34_cleanaux_s1.pt
  SubEnsembleTTA ens7+D4TTA  -1          0.0440    0.0440      True                  (derived)


## Predictions in the nb19 schema (drop-in for the nb20 comparison plots)

In [7]:
TAG2 = '' if MODE == 'full' else '_smoke'
def region_name(r): return f'{int(PITCH[int(r)])}mm'
def rows_for(fam, seed_out, pe):
    te = Et[kte]
    return pd.DataFrame(dict(model=fam, dataset='minbias', seed=seed_out, split='test',
                             true_energy=te, pred_energy=pe, pred_bias=pe / te - 1.0,
                             region=REG[kte], region_name=[region_name(r) for r in REG[kte]],
                             ET=ETT[kte]))
FAMS = {'SubNetW4': [('nb32_W4_s0', 0), ('nb32_W4_s1', 1), ('nb32_W4_s2', 2),
                     ('nb34_pure_s3', 3), ('nb34_pure_s4', 4)],
        'SubNetW4CleanAux': [('nb34_cleanaux_s0', 0), ('nb34_cleanaux_s1', 1)]}
for fam, members in FAMS.items():
    parts = []
    for name, seed_out in members:
        if name not in names: continue
        k = names.index(name)
        aa, bb = np.polyfit(va_plain[k], y[kva], 1)
        parts.append(rows_for(fam, seed_out, np.exp(aa * te_plain[k] + bb)))
    df = pd.concat(parts, ignore_index=True)
    df.to_csv(OUT / f'minbias__{fam}{TAG2}.csv', index=False)
    print(f'wrote minbias__{fam}{TAG2}.csv ({len(df)} rows, {len(parts)} seeds)')
df = rows_for('SubEnsembleTTA', -1, pe_tta)
df.to_csv(OUT / f'minbias__SubEnsembleTTA{TAG2}.csv', index=False)
print(f'wrote minbias__SubEnsembleTTA{TAG2}.csv ({len(df)} rows)')

wrote minbias__SubNetW4.csv (54420 rows, 5 seeds)


wrote minbias__SubNetW4CleanAux.csv (21768 rows, 2 seeds)


wrote minbias__SubEnsembleTTA.csv (10884 rows)


## Old vs new: the mentor-update comparison

In [8]:
gh = pd.read_csv(OUT / 'minbias__GateHuber.csv'); gh = gh[gh.split == 'test']
blocks = [gh[gh.seed == s].reset_index(drop=True) for s in sorted(gh.seed.unique())]
n0 = min(len(b) for b in blocks)
te_old = blocks[0].true_energy.to_numpy()[:n0]
pe_old = np.stack([b.pred_energy.to_numpy()[:n0] for b in blocks]).mean(0)
print(f'old best (nb19 GateHuber 5-seed ens): {resolution(pe_old, te_old)["sigma_eff"]:.4f}')
print(f'new best (SubEnsembleTTA)           : {resolution(pe_tta, Et[kte])["sigma_eff"]:.4f}')
def perbin(pe, te):
    edges = np.quantile(te, np.linspace(0, 1, 7)); o = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te >= edges[i]) & (te < hi)
        o.append((f'{edges[i]:.0f}-{edges[i+1]:.0f}', resolution(pe[mm], te[mm])['sigma_eff'], int(mm.sum())))
    return o
print(f'{"E bin (GeV)":>12s} {"old":>8s} {"new":>8s} {"err(+-)":>8s}')
for (b1, s1, n1), (b2, s2, n2) in zip(perbin(pe_old, te_old), perbin(pe_tta, Et[kte])):
    print(f'{b2:>12s} {s1:8.4f} {s2:8.4f} {0.96 * s2 / np.sqrt(n2):8.4f}')

old best (nb19 GateHuber 5-seed ens): 0.0463
new best (SubEnsembleTTA)           : 0.0440
 E bin (GeV)      old      new  err(+-)
        2-11   0.0765   0.0668   0.0015
       11-17   0.0488   0.0499   0.0011
       17-24   0.0381   0.0369   0.0008
       24-34   0.0373   0.0382   0.0009
       34-53   0.0394   0.0366   0.0008
      53-100   0.0413   0.0374   0.0008
